# Importar librerias

In [ ]:
import pandas as pd
import numpy as np
import torch

from typing import Any, List, Optional, Dict
import json
import math
import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

variables

In [ ]:
ASEED = 42

TRAIN_CSV = "train_public.csv"
DEV_CSV = "dev_public.csv"
OUTPUT_SUBMISSION = "results.csv"

TITLE_COLS = [f"title_{i}" for i in range(1, 11)]
N_COLS = 10 # Los 10 títulos

# Cargar datos

In [ ]:
# Datos de descarga
DATASET_URL = "https://pln.inf.um.es/corpora/politicheadlines/2026/development_phase_dataset.zip"
ZIP_NAME = "development_phase_dataset.zip"

# Descarga del fichero
!wget -q $DATASET_URL -O $ZIP_NAME
print("Fichero descargado:", ZIP_NAME)

# Descomprimir fichero
!unzip -q $ZIP_NAME
print("Dataset descomprimido")

Fichero descargado: development_phase_dataset.zip
Dataset descomprimido


In [ ]:
# Función de validación de columnas
def validate_columns(df: pd.DataFrame) -> None:
    required = ["id", "article_body", "image_hash"] + TITLE_COLS
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {missing}\nPresentes: {list(df.columns)}")

df_train = pd.read_csv(TRAIN_CSV)
df_test= pd.read_csv(DEV_CSV)
validate_columns(df_train)
validate_columns(df_test)

train_df = df_train.copy()
test_df = df_test.copy()

display(df_train.head(3))

,id,article_body,image_hash,title_1,title_2,title_3,title_4,title_5,title_6,title_7,title_8,title_9,title_10,y_true
0,5ac211a9c567076b1d2ed1f20260e3ee34d63b9f9c131d...,A seis días de culminar su V Asamblea Ciudadan...,c4ed18e02fde9ebf95760b06dcb4180b,ERC y PSC afrontan la negociación de las cuent...,Podemos lanza a Irene Irene Montero para lider...,Sánchez vuelve a servirse de la Abogacía del E...,Un Madrid sobrenatural,Calviño logra los apoyos necesarios para presi...,Inquietud por la vivienda,El líder de la conservadora CDU repudia a la A...,El efecto Trump obliga a reconfigurar la polít...,Podemos lanza a Irene Montero para liderar una...,Ayuso pide formalmente a la Delegación del Gob...,t9 t2 t7 t5 t6 t10 t3 t8 t4 t1
1,7b1760706e43fad04875ffbf950a95b109330fb7fe8420...,La expectación reina entre las universidades d...,e9b7805ee2d126820b80ebfda0d29fdd,Manifestación en Palma contra el «negocio» de ...,"Resultado elecciones generales España 2023 , e...",El sector turístico impulsa corredores sanitar...,Figuras clave que renegaron de Trump reculan a...,"Las universidades privadas de prestigio, ante ...",Sin respuesta de la Universidad. Silencio ante...,Oriol Junqueras repetirá como presidente de ER...,EEUU reclama a España más gasto en defensa tra...,Malestar en Moncloa por el boicot de Podemos a...,Sánchez fija prioridades para un nuevo curso a...,t5 t3 t6 t2 t8 t4 t9 t1 t7 t10
2,f35db670b78794eac624cec7e6015e5bffe360510a6cb3...,Sánchez abre una nueva relación con Vietnam y ...,595490c0981c6fc1e7a23c6a091d7ee9,Qué interés comercial podría tener España en V...,Sánchez abre una nseguridad alimentariava rela...,PNV y BNG muestran su voluntad de acuerdo con ...,Pedro Sánchez viaja a China por tercera vez: d...,Sánchez abre una nueva relación con Vietnam y ...,Tamames: la verdad contra la propaganda,"García Montero, director del Cervantes: “La RA...",Feijóo: “Nos dirigimos a una profundísima cris...,Sánchez advierte en Vietnam contra los arancel...,Sánchez cree que el cambio de Casado a Feijóo ...,t5 t2 t9 t1 t6 t7 t10 t4 t3 t8


preparacion del head+tail

In [ ]:
# Separo validacion
df_train_base, df_val_base = train_test_split(train_df, test_size=0.1, random_state=ASEED)
print(f"Noticias - Train: {len(df_train_base)} | Val: {len(df_val_base)} | Test: {len(test_df)}")

# Funcion para aplicar el head tail
def aplicar_head_tail(texto, max_palabras=350):
    if pd.isnull(texto): return ""


    palabras = str(texto).split()
    if len(palabras) <= max_palabras: return texto
    mitad = max_palabras // 2
    return " ".join(palabras[:mitad]) + " ... [SALTO] ... " + " ".join(palabras[-mitad:])

print("Recortando artículos largos...")
for df in [df_train_base, df_val_base, test_df]:
    df['article_body'] = df['article_body'].apply(aplicar_head_tail)

# Melt para cros encoder
def transformar_a_formato_clasificacion(df):
    cols_fijas = ['id', 'article_body']
    if 'y_true' in df.columns:
        cols_fijas.append('y_true')

    df_melted = df.melt(
        id_vars=cols_fijas,
        value_vars=[f'title_{i}' for i in range(1, 11)],
        var_name='columna_original',
        value_name='title'
    )

    df_melted['token'] = df_melted['columna_original'].str.replace('title_', 't')

    if 'y_true' in df.columns:
        df_melted['winner'] = df_melted['y_true'].str.split().str[0]
        df_melted['label'] = (df_melted['token'] == df_melted['winner']).astype(int)
        df_melted = df_melted.drop(columns=['columna_original', 'winner'])
    else:
        df_melted = df_melted.drop(columns=['columna_original'])

    return df_melted.sort_values(by=['id', 'token']).reset_index(drop=True)

print("Transformando datasets a 10 filas por noticia (Melt)...")
train_ready = transformar_a_formato_clasificacion(df_train_base)
val_ready = transformar_a_formato_clasificacion(df_val_base)
test_ready = transformar_a_formato_clasificacion(test_df)

# Comprobación
display(train_ready[['id', 'token', 'label']].head(10))

Noticias - Train: 90 | Val: 10 | Test: 50
Recortando artículos largos...
Transformando datasets a 10 filas por noticia (Melt)...


,id,token,label
0,01f3cc2dbfa0351e046e6398305e64699c25287fe666f3...,t1,0
1,01f3cc2dbfa0351e046e6398305e64699c25287fe666f3...,t10,1
2,01f3cc2dbfa0351e046e6398305e64699c25287fe666f3...,t2,0
3,01f3cc2dbfa0351e046e6398305e64699c25287fe666f3...,t3,0
4,01f3cc2dbfa0351e046e6398305e64699c25287fe666f3...,t4,0
5,01f3cc2dbfa0351e046e6398305e64699c25287fe666f3...,t5,0
6,01f3cc2dbfa0351e046e6398305e64699c25287fe666f3...,t6,0
7,01f3cc2dbfa0351e046e6398305e64699c25287fe666f3...,t7,0
8,01f3cc2dbfa0351e046e6398305e64699c25287fe666f3...,t8,0
9,01f3cc2dbfa0351e046e6398305e64699c25287fe666f3...,t9,0


In [ ]:
# Balanceo de clases
def balancear_dataset(df, ceros_por_noticia=2):
    print(f"Tamaño original: {len(df)} filas.")

    df_unos = df[df['label'] == 1]
    df_ceros = df[df['label'] == 0]

    df_ceros_reducido = df_ceros.groupby('id').sample(n=ceros_por_noticia, random_state=42, replace=True)

    df_balanceado = pd.concat([df_unos, df_ceros_reducido])

    df_balanceado = df_balanceado.sort_values(by=['id', 'token']).reset_index(drop=True)

    df_balanceado = df_balanceado.drop_duplicates()

    print(f"Tamaño balanceado: {len(df_balanceado)} filas.")
    return df_balanceado

train_balanceado = balancear_dataset(train_ready, ceros_por_noticia=2)
display(train_balanceado.head(6))

Tamaño original: 900 filas.
Tamaño balanceado: 262 filas.


,id,article_body,y_true,title,token,label
0,01f3cc2dbfa0351e046e6398305e64699c25287fe666f3...,EH Bildu ha reelegido este sábado a Arnaldo Ot...,t10 t3 t7 t1 t6 t5 t9 t4 t8 t2,Arnaldo Otegi: “Estamos preparados para llegar...,t10,1
1,01f3cc2dbfa0351e046e6398305e64699c25287fe666f3...,EH Bildu ha reelegido este sábado a Arnaldo Ot...,t10 t3 t7 t1 t6 t5 t9 t4 t8 t2,De «Mr. Marshall» a «Mr. No»: ¿cómo será ahora...,t4,0
2,01f3cc2dbfa0351e046e6398305e64699c25287fe666f3...,EH Bildu ha reelegido este sábado a Arnaldo Ot...,t10 t3 t7 t1 t6 t5 t9 t4 t8 t2,Feijóo pide su confianza a los líderes europeo...,t7,0
3,0425349f5d4ef6866b0e11651adc5965da80fa741965f7...,Los toreros están acostumbrados a vivir moment...,t8 t6 t7 t9 t5 t1 t2 t3 t4 t10,La liga francesa suspende a Mbappé tres partid...,t4,0
4,0425349f5d4ef6866b0e11651adc5965da80fa741965f7...,Los toreros están acostumbrados a vivir moment...,t8 t6 t7 t9 t5 t1 t2 t3 t4 t10,El Supremo insta al Gobierno a incluir los tor...,t7,0
5,0425349f5d4ef6866b0e11651adc5965da80fa741965f7...,Los toreros están acostumbrados a vivir moment...,t8 t6 t7 t9 t5 t1 t2 t3 t4 t10,Una iniciativa legislativa popular obligará al...,t8,1


# Tokenizacion


In [ ]:
# Convierto a hf dataset
hf_train = Dataset.from_pandas(train_balanceado)
hf_val = Dataset.from_pandas(val_ready)
hf_test = Dataset.from_pandas(test_ready)

# Cargo el tokenizador
MODEL_NAME = "microsoft/mdeberta-v3-base"
print(f"Cargando tokenizador: {MODEL_NAME}...")
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
except:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

# Funcion para tokenizacion
def tokenize_batch(examples):
    # Le doy contexto
    textos_guiados = ["Noticia: " + str(t) for t in examples["article_body"]]
    titulos_guiados = ["¿Es este su titular? " + str(t) for t in examples["title"]]

    return tokenizer(
        textos_guiados,
        titulos_guiados,
        padding="max_length",
        truncation=True,
        max_length=512
    )

# Aplico todo
print("Tokenizando datos...")
tokenized_train = hf_train.map(tokenize_batch, batched=True)
tokenized_val = hf_val.map(tokenize_batch, batched=True)
tokenized_test = hf_test.map(tokenize_batch, batched=True)

if "label" in tokenized_train.column_names:
    tokenized_train = tokenized_train.rename_column("label", "labels")

if "label" in tokenized_val.column_names:
    tokenized_val = tokenized_val.rename_column("label", "labels")

if "label" in tokenized_test.column_names:
    tokenized_test = tokenized_test.rename_column("label", "labels")

print("¡Tokenización completada!")

Cargando tokenizador: microsoft/mdeberta-v3-base...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

Tokenizando datos...


Map:   0%|          | 0/262 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

¡Tokenización completada!


In [ ]:
display(pd.DataFrame(tokenized_train[:5]))

,id,article_body,y_true,title,token,labels,__index_level_0__,input_ids,token_type_ids,attention_mask
0,01f3cc2dbfa0351e046e6398305e64699c25287fe666f3...,EH Bildu ha reelegido este sábado a Arnaldo Ot...,t10 t3 t7 t1 t6 t5 t9 t4 t8 t2,Arnaldo Otegi: “Estamos preparados para llegar...,t10,1,0,"[260, 136440, 268, 89761, 13321, 274, 561, 101...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,01f3cc2dbfa0351e046e6398305e64699c25287fe666f3...,EH Bildu ha reelegido este sábado a Arnaldo Ot...,t10 t3 t7 t1 t6 t5 t9 t4 t8 t2,De «Mr. Marshall» a «Mr. No»: ¿cómo será ahora...,t4,0,1,"[260, 136440, 268, 89761, 13321, 274, 561, 101...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,01f3cc2dbfa0351e046e6398305e64699c25287fe666f3...,EH Bildu ha reelegido este sábado a Arnaldo Ot...,t10 t3 t7 t1 t6 t5 t9 t4 t8 t2,Feijóo pide su confianza a los líderes europeo...,t7,0,2,"[260, 136440, 268, 89761, 13321, 274, 561, 101...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,0425349f5d4ef6866b0e11651adc5965da80fa741965f7...,Los toreros están acostumbrados a vivir moment...,t8 t6 t7 t9 t5 t1 t2 t3 t4 t10,La liga francesa suspende a Mbappé tres partid...,t4,0,3,"[260, 136440, 268, 2714, 289, 34662, 264, 1958...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,0425349f5d4ef6866b0e11651adc5965da80fa741965f7...,Los toreros están acostumbrados a vivir moment...,t8 t6 t7 t9 t5 t1 t2 t3 t4 t10,El Supremo insta al Gobierno a incluir los tor...,t7,0,4,"[260, 136440, 268, 2714, 289, 34662, 264, 1958...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


# Carga del modelo

In [ ]:
# Cargo el modelo
print(f"Cargando modelo: {MODEL_NAME}...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True
)

# Métricas para el entrenamiento
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Calculo las métricas
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='binary')

    return {"accuracy": acc, "f1": f1}

class EntrenadorCastigador(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        import torch

        # Sacamos las etiquetas correctas
        labels = inputs.pop("labels")
        # Hacemos la predicción
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # --- EL FIX ---
        # Obligamos a los pesos a tener el MISMO formato (Half/Float16) que las predicciones
        pesos_clase = torch.tensor([1.0, 8.0], device=model.device, dtype=logits.dtype)

        loss_fct = torch.nn.CrossEntropyLoss(weight=pesos_clase)

        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

# Hiperparamettos
training_args = TrainingArguments(
    output_dir="./deberta_v3_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=False,
    fp16=False,
    num_train_epochs=5,
    weight_decay=0.02,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=1,
    logging_steps=10
)

# Trainer
trainer = EntrenadorCastigador(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("\n¡Trainer ensamblado!")

Cargando modelo: microsoft/mdeberta-v3-base...


pytorch_model.bin:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.bias                | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                  


¡Trainer ensamblado!


In [ ]:
from collections import Counter

# Contamos las etiquetas en tu dataset de entrenamiento
conteos_train = Counter(tokenized_train['labels'])
total_train = len(tokenized_train)

print("RECUENTO EN EL DATASET DE ENTRENAMIENTO (tokenized_train):")
print(f"Clase 0 (Falsos): {conteos_train[0]} ({(conteos_train[0]/total_train)*100:.1f}%)")
print(f"Clase 1 (Verdaderos): {conteos_train[1]} ({(conteos_train[1]/total_train)*100:.1f}%)")

print("\n--------------------------------------------------")

# Ya que estamos, comprobamos también el de validación por si acaso
conteos_val = Counter(tokenized_val['labels'])
total_val = len(tokenized_val)
print("RECUENTO EN EL DATASET DE VALIDACIÓN (tokenized_val):")
print(f"Clase 0 (Falsos): {conteos_val[0]} ({(conteos_val[0]/total_val)*100:.1f}%)")
print(f"Clase 1 (Verdaderos): {conteos_val[1]} ({(conteos_val[1]/total_val)*100:.1f}%)")

RECUENTO EN EL DATASET DE ENTRENAMIENTO (tokenized_train):
Clase 0 (Falsos): 172 (65.6%)
Clase 1 (Verdaderos): 90 (34.4%)

--------------------------------------------------
RECUENTO EN EL DATASET DE VALIDACIÓN (tokenized_val):
Clase 0 (Falsos): 90 (90.0%)
Clase 1 (Verdaderos): 10 (10.0%)


## optuna


In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 16.4 MB/s eta 0:00:00


In [ ]:
import optuna

print(f"Buscando hiperparámetros...")

# Función de inicialización
def model_init():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        ignore_mismatched_sizes=True
    )

# Espacio de Búsqueda
def optuna_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 5e-6, 4e-5, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.1),
        "warmup_ratio": trial.suggest_float("warmup_ratio", 0.0, 0.2),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 3, 10)
    }

# Argumentos Fijos
training_args = TrainingArguments(
    output_dir="./optuna_search_full",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=False,
    fp16=False,
    max_grad_norm=1.0,
    adam_epsilon=1e-6,
    logging_steps=20,
    disable_tqdm=True,
    load_best_model_at_end=True,
    disable_tqdm=False,
    metric_for_best_model="f1"
)

# Ensamblo el trainer
trainer_optuna = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Iniciando Optuna...")
best_run = trainer_optuna.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=optuna_hp_space,
    n_trials=10,
    compute_objective=lambda metrics: metrics["eval_f1"]
)

# RESULTADO FINAL
print("\n" + "="*50)
print("¡BÚSQUEDA TERMINADA! LA MEJOR CONFIGURACIÓN ES:")
print(best_run.hyperparameters)
print("="*50)

Buscando hiperparámetros...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
mask_predictions.classifier.bias           | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                  

Iniciando Optuna...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

...

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
mask_predictions.classifier.bias           | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                  

{'loss': '3.322', 'grad_norm': '20.3', 'learning_rate': '1.587e-05', 'epoch': '0.3556'}
{'loss': '3.062', 'grad_norm': '6.852', 'learning_rate': '1.504e-05', 'epoch': '0.7111'}
{'eval_loss': '0.3387', 'eval_accuracy': '0.9', 'eval_f1': '0', 'eval_runtime': '2.082', 'eval_samples_per_second': '48.04', 'eval_steps_per_second': '24.02', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[W 2026-03-17 09:16:21,614] Trial 1 failed with parameters: {'learning_rate': 1.666480040896488e-05, 'weight_decay': 0.06795119534229023, 'warmup_ratio': 0.12381492022787002, 'num_train_epochs': 7} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/integrations/integration_utils.py", line 253, in _objective
    trainer.train(resume_from_checkpoint=checkpoint, trial=trial)
  File "/usr/local/lib/python3.12/dist-packages/transformers/trainer.py", line 2174, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/trainer.py", line 2641, in _inner_training_loop
    self._maybe_log_save_evaluate(
  File "/usr/local/lib/python3.12/dist-packages/transform

KeyboardInterrupt: 

# Entrenamiento

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,5.705518,0.626465,0.900000,0.000000
2,6.485870,0.986328,0.100000,0.181818
3,5.563770,0.661133,0.900000,0.000000
4,5.182910,0.729004,0.100000,0.181818


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

TrainOutput(global_step=68, training_loss=5.514336978687959, metrics={'train_runtime': 199.7285, 'train_samples_per_second': 6.559, 'train_steps_per_second': 0.426, 'total_flos': 275745331101696.0, 'train_loss': 5.514336978687959, 'epoch': 4.0})

# Resultados

funciones auxiliares

In [ ]:
def _parse_rank_list(x: Any) -> List[str]:
    """
    Acepta:
      - "t1 t7 t3" (espacios)
      - "t1,t7,t3" (comas)
      - '["t1","t7"]' (json list)
      - "t1" (top-1)
    """
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    s = str(x).strip()
    if not s:
        return []

    if s.startswith("[") and s.endswith("]"):
        try:
            arr = json.loads(s)
            if isinstance(arr, list):
                return [str(t).strip() for t in arr if str(t).strip()]
        except Exception:
            pass

    s = s.replace("\t", " ").replace("\n", " ").replace("\r", " ").replace(";", " ")
    if "," in s:
        parts = [p.strip() for p in s.split(",")]
    else:
        parts = [p.strip() for p in s.split()]
    return [p for p in parts if p]

def _token_to_col(tok: Any) -> Optional[int]:
    """
    't7' -> 7
    """
    if tok is None or (isinstance(tok, float) and pd.isna(tok)):
        return None
    s = str(tok).strip()
    if len(s) < 2:
        return None
    prefix = s[0].lower()
    if prefix not in ("t", "d"):
        return None
    try:
        return int(s[1:])
    except Exception:
        return None

def _unique_valid_pred_cols(pred: List[str], n_cols: int) -> List[int]:
    """
    Convierte tokens a columnas 1..n_cols, elimina duplicados preservando orden.
    """
    out: List[int] = []
    seen = set()
    for tok in pred:
        n = _token_to_col(tok)
        if n is None or not (1 <= n <= n_cols):
            continue
        if n in seen:
            continue
        seen.add(n)
        out.append(n)
    return out

def _ndcg_from_ideal(pred_cols: List[int], ideal_cols: List[int], k: int) -> float:
    """
    nDCG@k donde el ideal es un ranking explícito ideal_cols.
    Ganancia lineal por posición ideal: top=10..1 (si len=10).
    """
    if not ideal_cols:
        return 0.0

    ideal_rank: Dict[int, int] = {c: i for i, c in enumerate(ideal_cols)}

    def gain_for_col(c: int) -> float:
        r = ideal_rank.get(c, None)
        if r is None:
            return 0.0
        return float(len(ideal_cols) - r)

    dcg = 0.0
    for i, c in enumerate(pred_cols[:k], start=1):
        dcg += gain_for_col(c) / math.log2(i + 1)

    idcg = 0.0
    for i, c in enumerate(ideal_cols[:k], start=1):
        idcg += gain_for_col(c) / math.log2(i + 1)

    if idcg <= 0.0:
        return 0.0
    return max(0.0, min(1.0, dcg / idcg))

def pa_ndcg(pred_tokens: List[str], true_tokens: List[str], k: int = 10, alpha: float = 0.9) -> float:
    """
    PA-nDCG:
      - Si top-1 no coincide -> 0
      - Si coincide -> alpha + (1-alpha)*aux_nDCG(resto)
    """
    if not pred_tokens or not true_tokens:
        return 0.0

    ideal_cols = _unique_valid_pred_cols(true_tokens, N_COLS)
    pred_cols  = _unique_valid_pred_cols(pred_tokens, N_COLS)

    if not ideal_cols or not pred_cols:
        return 0.0

    if pred_cols[0] != ideal_cols[0]:
        return 0.0

    primary = ideal_cols[0]
    pred_rest  = [c for c in pred_cols if c != primary]
    ideal_rest = [c for c in ideal_cols if c != primary]

    aux = _ndcg_from_ideal(pred_rest, ideal_rest, k=k)
    score = alpha + (1.0 - alpha) * aux
    return max(0.0, min(1.0, score))

In [ ]:
def evaluar_pa_ndcg_en_validacion(trainer, dataset_tokenizado, df_original_val):
    print("Haciendo predicciones en Validación...")
    predicciones = trainer.predict(dataset_tokenizado)

    # --- EL FIX AQUÍ ---
    # Convertimos explícitamente a float32 para que Pandas no explote
    import torch
    logits = torch.tensor(predicciones.predictions)
    scores = torch.nn.functional.softmax(logits, dim=-1)[:, 1].numpy().astype("float32")

    # 2. Asociamos los scores al dataframe 'melted'
    # IMPORTANTE: Asegúrate de que 'val_ready' es el DF que usaste para tokenizar
    df_eval = val_ready.copy()
    df_eval['model_score'] = scores

    # 3. Reconstruimos el ranking: ordenamos por ID de noticia y probabilidad descendente
    df_ordenado = df_eval.sort_values(by=['id', 'model_score'], ascending=[True, False])

    # 4. Agrupamos los tokens en un string separado por espacios
    pred_rankings = df_ordenado.groupby('id')['token'].apply(lambda x: ' '.join(x)).reset_index()
    pred_rankings.rename(columns={'token': 'y_pred'}, inplace=True)

    # 5. Cruzamos con las respuestas correctas
    df_comparativo = df_original_val[['id', 'y_true']].merge(pred_rankings, on='id')

    # 6. Calculamos el PA-nDCG
    lista_scores = []
    for _, row in df_comparativo.iterrows():
        p_tokens = _parse_rank_list(row['y_pred'])
        t_tokens = _parse_rank_list(row['y_true'])

        # Usamos tus funciones de métrica
        score_i = pa_ndcg(p_tokens, t_tokens, k=10, alpha=0.9)
        lista_scores.append(score_i)

    nota_media = np.mean(lista_scores)

    print("\n" + "="*50)
    print(f"NOTA PA-nDCG EN VALIDACIÓN: {nota_media:.4f}")
    print("="*50)

    return df_comparativo

# Ejecuta de nuevo
resultados_val = evaluar_pa_ndcg_en_validacion(trainer, tokenized_val, df_val_base)

Haciendo predicciones en Validación...



NOTA PA-nDCG EN VALIDACIÓN: 0.0975
